# 03 — Parent → child spawn with lineage

A parent task can spawn a child task that materializes a fresh notebook from a template, runs against the **parent's remaining budget reservation**, and feeds its result back to the parent. The spawn manager enforces `max_spawn_depth`, `max_children_per_task`, and `parent_reserve_floor_ratio` from the active policy profile.

This example shows the parent driving from outside the kernel (Python API) for clarity; the same call works from inside a running notebook — the M5 integration test (`test_m5_spawn.py`) exercises that path.

**Kernel:** standard `python3`.

In [ ]:
import json, tempfile, shutil
from pathlib import Path

import nbformat
from nbformat.v4 import new_notebook, new_code_cell

from agent_kernel.api import AgentKernel
from agent_kernel.models.task import SpawnSpec
from agent_kernel.runtime.template_registry import list_templates

workspace = Path(tempfile.mkdtemp(prefix='ak-ex03-'))
ak = AgentKernel(workspace)
print('templates available:', list_templates())

## 1. Create the parent task

The parent's notebook can be anything — for this demo it's a trivial one-cell notebook; the interesting work happens in the child spawn we issue against it.

In [ ]:
parent_nb = workspace / 'parent.ipynb'
nbformat.write(new_notebook(
    cells=[new_code_cell('print("parent setup")')],
    metadata={'kernelspec': {'name': 'python3', 'display_name': 'Python 3'}},
), parent_nb)

parent = ak.create_task(notebook_path=str(parent_nb), kernel_name='python3')
print('parent task:', parent.task_id, 'depth:', parent.depth)

## 2. Spawn a child from the `python-analysis` template

The template ships with the package (`agent_kernel/templates/python-analysis.ipynb`) and accepts `query` and `limit` parameters. The materializer injects them via the two-channel model: an executable parameter cell (Python) **and** a metadata block at `metadata.agent_kernel.inputs` for non-Python kernels.

In [ ]:
spawn_result = ak.spawn_child_task(
    parent.task_id,
    SpawnSpec(
        template_name='python-analysis',
        parameters={'query': 'recent papers', 'limit': 5},
        kernel_name='python3',
    ),
)
print('allowed:        ', spawn_result.allowed)
print('reason:         ', spawn_result.reason)
child = spawn_result.child_task
print('child task:     ', child.task_id)
print('  parent:       ', child.parent_task_id)
print('  spawn_index:  ', child.spawn_index)
print('  depth:        ', child.depth)
print('  reserved:     ', child.reserved_budget.model_dump())
print('  materialized: ', child.notebook_path)

## 3. Run the child

`spawn_child_task` only materializes and reserves; the caller runs the child explicitly. This keeps spawn pure (no I/O beyond materialize + ledger emit) and lets the parent decide whether to await or fire-and-forget.

In [ ]:
final_child = ak.run_task(child.task_id)
print('child final:', final_child.status.value)
print('executed:   ', final_child.executed_notebook_path)

## 4. The full lineage chain in the JSONL

Walking events filtered to **both** task ids shows the canonical chain:

```
task.created (parent)
task.spawn.requested (parent)
notebook.materialized (child)
task.created (child)
task.spawned (parent, references child)
notebook.execution.started (child)
cell.execution.* …
notebook.execution.completed (child)
task.completed (child)
```


In [ ]:
ids = {parent.task_id, child.task_id}
for e in ak.list_events():
    if e.task_id in ids:
        print(f'{e.event_type.value:35s}  task={e.task_id}')

## 5. Verify the injected parameters made it into the child

The materializer wrote `query` and `limit` into the notebook's metadata; the Python parameter injector also wrote a `# Injected by agent-kernel` cell. We can read both from the executed notebook.

In [ ]:
executed = nbformat.read(final_child.executed_notebook_path, as_version=4)
print('metadata.agent_kernel.inputs:', json.dumps(executed.metadata.get('agent_kernel', {}).get('inputs', {}), indent=2))
print()
for c in executed.cells:
    if 'injected-parameters' in (c.metadata.get('tags') or []):
        print('--- injected cell ---')
        print(c.source)
        break

In [ ]:
shutil.rmtree(workspace, ignore_errors=True)